<a href="https://colab.research.google.com/github/Joan2022Laurente/fade/blob/main/notebooks/zImageTurboV2.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
# ================================================================
#   SETUP — Dark Beast (Z-Image Turbo) + Qwen3-4B + VAE + LoRA
#   + modelos custom · descarga paralela con aria2c (16 conexiones)
#   + ControlNet Union (inpainting workflow)
# ================================================================

# ── FORMULARIO ─────────────────────────────────────────────────

# --- Checkpoint 1 ---
CKPT1_URL  = "https://civitai.red/api/download/models/2957651?fileId=2837007"  # @param {type:"string"}
CKPT1_NAME = "divingZimageTurbo.safetensors"                                   # @param {type:"string"}

# --- Checkpoint 2 ---
CKPT2_URL  = "https://civitai.red/api/download/models/2918406?fileId=2804511"  # @param {type:"string"}
CKPT2_NAME = "prnmasterBASE.safetensors"                                        # @param {type:"string"}

# --- Checkpoint 3 ---
CKPT3_URL  = "https://civitai.red/api/download/models/2754977?fileId=2641972"  # @param {type:"string"}
CKPT3_NAME = "RedCraftERNIERedMIX.safetensors"                                  # @param {type:"string"}

# --- Checkpoint 4 ---
CKPT4_URL  = "https://civitai.red/api/download/models/2668773?fileId=2555711"  # @param {type:"string"}
CKPT4_NAME = "byStable_Yogi.safetensors"                                        # @param {type:"string"}

# --- LoRA HuggingFace (fijo) ---
LORA_HF_ENABLED = True  # @param {type:"boolean"}

# --- LoRA Civitai 1 ---
LORA1_URL  = ""                                                                 # @param {type:"string"}
LORA1_NAME = "zitMystic.safetensors"                                            # @param {type:"string"}

# --- LoRA Civitai 2 ---
LORA2_URL  = "https://civitai.red/api/download/models/2486059?fileId=2374406"  # @param {type:"string"}
LORA2_NAME = "nsMASTER.safetensors"                                             # @param {type:"string"}

# --- LoRA Civitai 3 ---
LORA3_URL  = "https://civitai.red/api/download/models/2454516?fileId=2344803"  # @param {type:"string"}
LORA3_NAME = "povor4.safetensors"                                               # @param {type:"string"}

# --- LoRA Civitai 4 ---
LORA4_URL  = "https://civitai.red/api/download/models/2643568?fileId=2531431"  # @param {type:"string"}
LORA4_NAME = "ahe.safetensors"                                                  # @param {type:"string"}

# --- ControlNet Union (inpainting workflow gsl_creator_2) ---
CONTROLNET_ENABLED     = True   # @param {type:"boolean"}  → descarga model patch + workflow
CONTROLNET_SKIP_COMPAT = False  # @param {type:"boolean"}  → True = omite la verificación de compatibilidad

# ───────────────────────────────────────────────────────────────
import os, subprocess, sys, shutil
from google.colab import userdata

CIVITAI_KEY   = userdata.get('CIVITAI_KEY')
TAILSCALE_KEY = userdata.get('TAILSCALE_KEY')
HF_TOKEN      = userdata.get('HF_TOKEN')
if not CIVITAI_KEY:   raise RuntimeError("❌ Falta CIVITAI_KEY")
if not TAILSCALE_KEY: raise RuntimeError("❌ Falta TAILSCALE_KEY")
if not HF_TOKEN:      raise RuntimeError("❌ Falta HF_TOKEN")
print("✅ Secrets cargados")

COMFY        = "/content/ComfyUI"
MODELS       = f"{COMFY}/models"
CUSTOM_NODES = f"{COMFY}/custom_nodes"
WORKFLOWS    = f"{COMFY}/user/default/workflows"
HF_Z         = "https://huggingface.co/Comfy-Org/z_image_turbo/resolve/main/split_files"

# ── Helpers base ───────────────────────────────────────────────
def run(cmd, cwd=None):
    r = subprocess.run(cmd, shell=True, cwd=cwd, text=True,
                       stdout=subprocess.PIPE, stderr=subprocess.STDOUT)
    if r.stdout.strip(): print(r.stdout.strip()[-1500:])
    return r.returncode

def pip(pkg):
    run(f'"{sys.executable}" -m pip install {pkg} -q --no-deps 2>/dev/null || '
        f'"{sys.executable}" -m pip install {pkg} -q')

def clone(url, dst):
    if os.path.isdir(dst): run("git pull --ff-only", cwd=dst)
    else:                   run(f'git clone --depth 1 "{url}" "{dst}"')
    req = f"{dst}/requirements.txt"
    if os.path.exists(req): run(f'"{sys.executable}" -m pip install -r "{req}" -q')

# ── Descarga con aria2c (16 conexiones paralelas) ──────────────
def dl(url, folder, name, auth_header=None):
    os.makedirs(folder, exist_ok=True)
    dest = f"{folder}/{name}"

    if os.path.exists(dest) and os.path.getsize(dest) > 1_048_576:
        print(f"  ✅ {name} ya existe — skip")
        return dest

    print(f"  ↓ {name}  [aria2c · 16 conexiones]")
    auth = f'--header="Authorization: Bearer {auth_header}"' if auth_header else ""
    cmd  = (f'aria2c -c -x 16 -s 16 -k 1M '
            f'--console-log-level=error --summary-interval=0 '
            f'{auth} -d "{folder}" -o "{name}" "{url}"')
    run(cmd)

    ok = os.path.exists(dest) and os.path.getsize(dest) > 1_048_576
    sz = os.path.getsize(dest) / 1024**3 if ok else 0
    print(f"  {'✅' if ok else '❌'} {name}  ({sz:.2f} GB)")
    return dest if ok else None

def dl_hf(url, folder, name):
    return dl(url, folder, name, auth_header=HF_TOKEN)

def dl_civitai(url, folder, name):
    sep = "&" if "?" in url else "?"
    return dl(f"{url}{sep}token={CIVITAI_KEY}", folder, name)

# ── SISTEMA ────────────────────────────────────────────────────
print("\n[SYS] Herramientas...")
run("apt-get update -qq && apt-get install -y -qq aria2 git wget curl")

# ── COMFYUI ────────────────────────────────────────────────────
print("\n[0] ComfyUI...")
if not os.path.isdir(COMFY):
    run(f'git clone --depth 1 https://github.com/comfyanonymous/ComfyUI.git "{COMFY}"')
else:
    run("git pull --ff-only", cwd=COMFY)
run(f'"{sys.executable}" -m pip install -r "{COMFY}/requirements.txt" -q')
for d in [f"{MODELS}/diffusion_models", f"{MODELS}/text_encoders",
          f"{MODELS}/vae", f"{MODELS}/loras", f"{MODELS}/model_patches",
          CUSTOM_NODES, WORKFLOWS]:
    os.makedirs(d, exist_ok=True)

# ── CUSTOM NODES ───────────────────────────────────────────────
print("\n[1] rgthree-comfy...")
clone("https://github.com/rgthree/rgthree-comfy.git", f"{CUSTOM_NODES}/rgthree-comfy")

print("\n[1.5] azzia-nodes...")
clone("https://github.com/Joan2022Laurente/node.git", f"{CUSTOM_NODES}/azzia-nodes")

# ── DEPS ───────────────────────────────────────────────────────
print("\n[2] Deps...")
for p in ["accelerate", "einops", "torchvision", "Pillow"]: pip(p)
print("  ✅")

# ── MODELOS FIJOS (encoder + VAE) ─────────────────────────────
print("\n[3] Text encoder + VAE...")
dl_hf(f"{HF_Z}/text_encoders/qwen_3_4b.safetensors",
      f"{MODELS}/text_encoders", "qwen_3_4b.safetensors")
dl_hf(f"{HF_Z}/vae/ae.safetensors",
      f"{MODELS}/vae", "ae.safetensors")

# ── CHECKPOINTS ───────────────────────────────────────────────
print("\n[4] Checkpoints...")
CHECKPOINTS = [
    (CKPT1_URL, CKPT1_NAME),
    (CKPT2_URL, CKPT2_NAME),
    (CKPT3_URL, CKPT3_NAME),
    (CKPT4_URL, CKPT4_NAME),
]
for i, (url, name) in enumerate(CHECKPOINTS, 1):
    if url.strip():
        print(f"  → Checkpoint {i}: {name}")
        dl_civitai(url.strip(), f"{MODELS}/diffusion_models", name.strip())
    else:
        print(f"  ⚪ Checkpoint {i}: vacío — skip")

# ── LoRAs ─────────────────────────────────────────────────────
print("\n[5] LoRAs...")
if LORA_HF_ENABLED:
    print("  → LoRA HF: YummyHDzit.safetensors")
    dl_hf("https://huggingface.co/joanjlau/loraprueba0/resolve/main/YummyHDzit.safetensors",
          f"{MODELS}/loras", "YummyHDzit.safetensors")
else:
    print("  ⚪ LoRA HF: desactivado — skip")

LORAS = [
    (LORA1_URL, LORA1_NAME),
    (LORA2_URL, LORA2_NAME),
    (LORA3_URL, LORA3_NAME),
    (LORA4_URL, LORA4_NAME),
]
for i, (url, name) in enumerate(LORAS, 1):
    if url.strip():
        print(f"  → LoRA Civitai {i}: {name}")
        dl_civitai(url.strip(), f"{MODELS}/loras", name.strip())
    else:
        print(f"  ⚪ LoRA Civitai {i}: vacío — skip")

# ── CONTROLNET UNION (inpainting workflow) ────────────────────
print("\n[6] ControlNet Union (inpainting)...")
if CONTROLNET_ENABLED:
    CN_NAME = "Z-Image-Turbo-Fun-Controlnet-Union-2.1.safetensors"
    CN_URL  = ("https://huggingface.co/alibaba-pai/Z-Image-Turbo-Fun-Controlnet-Union-2.1"
               f"/resolve/main/{CN_NAME}")

    if CONTROLNET_SKIP_COMPAT:
        print("  ⚡ CONTROLNET_SKIP_COMPAT=True — omitiendo verificación de compatibilidad")
    else:
        # Verificación de compatibilidad: ComfyUI >= 0.3.x soporta model_patches
        comfy_ver_file = f"{COMFY}/comfy/version.py"
        if os.path.exists(comfy_ver_file):
            with open(comfy_ver_file) as f:
                ver_content = f.read()
            print(f"  🔍 ComfyUI version.py encontrado — verificando...")
            # Comprueba que exista la carpeta model_patches como indicador de soporte
            patch_dir = f"{MODELS}/model_patches"
            if os.path.isdir(patch_dir):
                print("  ✅ Compatibilidad OK (model_patches/ existe)")
            else:
                print("  ⚠️  model_patches/ no encontrada — tu ComfyUI puede no soportar este modelo")
                print("      Actualiza ComfyUI o activa CONTROLNET_SKIP_COMPAT=True para forzar la descarga")
        else:
            print("  ⚠️  No se pudo verificar la versión de ComfyUI — continuando de todas formas")

    print(f"  → Descargando model patch: {CN_NAME}")
    dl_hf(CN_URL, f"{MODELS}/model_patches", CN_NAME)

    # Copiar el workflow gsl_creator_2.json si existe en azzia-nodes
    wf_src = f"{CUSTOM_NODES}/azzia-nodes/gsl_creator_2.json"
    wf_dst = f"{WORKFLOWS}/gsl_creator_2.json"
    if os.path.exists(wf_src):
        shutil.copy2(wf_src, wf_dst)
        print("  ✅ Workflow gsl_creator_2.json copiado desde azzia-nodes")
    else:
        print("  ⚠️  gsl_creator_2.json no encontrado en azzia-nodes — cópialo manualmente a:")
        print(f"      {wf_dst}")
else:
    print("  ⚪ ControlNet Union: desactivado — skip")

# ── WORKFLOWS (examples) ─────────────────────────────────────
print("\n[7] Workflows T2I...")

workflows = [
    "ZIMAGET2I_lora.json",
    "ZIMAGET2I_ImageHiresFix.json",
    "ZIMAGET2I_lora_hirexFix.json"
]

for wf in workflows:
    src = f"{CUSTOM_NODES}/azzia-nodes/{wf}"

    if os.path.exists(src):
        shutil.copy2(src, f"{WORKFLOWS}/{wf}")
        print(f"  ✅ {wf} copiado automáticamente desde azzia-nodes")
    else:
        print(f"  ⚠️ No se encontró {wf} en azzia-nodes")

# ── VERIFICACIÓN ───────────────────────────────────────────────
print("\n=== VERIFICACIÓN ===")

print("\n  — Fijos —")
for path, label in [
    (f"{MODELS}/text_encoders/qwen_3_4b.safetensors", "Qwen3-4B encoder"),
    (f"{MODELS}/vae/ae.safetensors",                  "Flux VAE"),
    (f"{CUSTOM_NODES}/rgthree-comfy",                 "rgthree"),
    (f"{CUSTOM_NODES}/azzia-nodes",                   "azzia-nodes"),
]:
    ok = os.path.exists(path) and (not os.path.isfile(path) or os.path.getsize(path) > 1_048_576)
    print(f"  {'✅' if ok else '❌'} {label}")

print("\n  — Checkpoints —")
for i, (url, name) in enumerate(CHECKPOINTS, 1):
    if url.strip():
        path = f"{MODELS}/diffusion_models/{name}"
        ok   = os.path.exists(path) and os.path.getsize(path) > 1_048_576
        print(f"  {'✅' if ok else '❌'} Checkpoint {i}: {name}")
    else:
        print(f"  ⚪ Checkpoint {i}: no configurado")

print("\n  — LoRAs —")
if LORA_HF_ENABLED:
    path = f"{MODELS}/loras/YummyHDzit.safetensors"
    ok   = os.path.exists(path) and os.path.getsize(path) > 1_048_576
    print(f"  {'✅' if ok else '❌'} LoRA HF: YummyHDzit.safetensors")
else:
    print("  ⚪ LoRA HF: desactivado")

for i, (url, name) in enumerate(LORAS, 1):
    if url.strip():
        path = f"{MODELS}/loras/{name}"
        ok   = os.path.exists(path) and os.path.getsize(path) > 1_048_576
        print(f"  {'✅' if ok else '❌'} LoRA Civitai {i}: {name}")
    else:
        print(f"  ⚪ LoRA Civitai {i}: no configurado")

print("\n  — ControlNet Union —")
if CONTROLNET_ENABLED:
    cn_path = f"{MODELS}/model_patches/Z-Image-Turbo-Fun-Controlnet-Union-2.1.safetensors"
    ok = os.path.exists(cn_path) and os.path.getsize(cn_path) > 1_048_576
    skip_tag = " [compat skipped]" if CONTROLNET_SKIP_COMPAT else ""
    print(f"  {'✅' if ok else '❌'} Z-Image-Turbo-Fun-Controlnet-Union-2.1{skip_tag}")
else:
    print("  ⚪ ControlNet Union: desactivado")

print("\n✅ SETUP COMPLETO")
print("→ KSampler: Steps=8 · CFG=1.0 · euler · beta")

✅ Secrets cargados

[SYS] Herramientas...
W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu jammy InRelease' does not seem to provide it (sources.list entry misspelt?)

[0] ComfyUI...
From https://github.com/comfyanonymous/ComfyUI
   08e93a3..ea73d3b  master     -> origin/master
Updating 08e93a3..ea73d3b
Fast-forward
 requirements.txt | 2 +-
 1 file changed, 1 insertion(+), 1 deletion(-)
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 14.5/14.5 MB 108.6 MB/s eta 0:00:00

[1] rgthree-comfy...
Already up to date.

[1.5] azzia-nodes...
Already up to date.

[2] Deps...
  ✅

[3] Text encoder + VAE...
  ✅ qwen_3_4b.safetensors ya existe — skip
  ✅ ae.safetensors ya existe — skip

[4] Checkpoints...
  → Checkpoint 1: divingZimageTurbo.safetensors
  ✅ divingZimageTurbo.safetensors ya existe — skip
  → Checkpoint 2: prnmasterBASE.safetensors
  ✅ prnmasterBASE.safetensors ya existe — skip
  → Checkpoint 3: RedCraftERNIERedMIX.safetensors
  

In [ ]:
# 🔒 Lanzar ComfyUI con Tailscale

import os, time
from google.colab import userdata

TAILSCALE_AUTH_KEY = userdata.get('TAILSCALE_KEY')
if not TAILSCALE_AUTH_KEY:
    raise RuntimeError("❌ Falta TAILSCALE_KEY en Secrets de Colab")

# 1. Instalar Tailscale
!curl -fsSL https://tailscale.com/install.sh | sh

# 2. Iniciar daemon
print("🚀 Iniciando tailscaled...")
!nohup tailscaled --tun=userspace-networking --socks5-server=localhost:1055 > /tmp/tailscaled.log 2>&1 &
time.sleep(5)

# 3. Verificar
print("🔍 Verificando tailscaled...")
!ps aux | grep tailscaled

# 4. Conectar
print("🔗 Conectando a Tailscale...")
!tailscale up --authkey={TAILSCALE_AUTH_KEY} --hostname=colab-comfyui

# 5. Mostrar IP
print("\n" + "="*50)
print("🌐 TU IP PRIVADA DE TAILSCALE ES:")
!tailscale ip -4
print("="*50)

# 6. Lanzar ComfyUI
import torch, gc
gc.collect()
torch.cuda.empty_cache()
print("\n🚀 Iniciando ComfyUI...")
%cd /content/ComfyUI
!python main.py --listen 0.0.0.0 --dont-print-server

Installing Tailscale for ubuntu jammy, using method apt
+ mkdir -p --mode=0755 /usr/share/keyrings
+ curl -fsSL https://pkgs.tailscale.com/stable/ubuntu/jammy.noarmor.gpg
+ tee /usr/share/keyrings/tailscale-archive-keyring.gpg
+ chmod 0644 /usr/share/keyrings/tailscale-archive-keyring.gpg
+ curl -fsSL https://pkgs.tailscale.com/stable/ubuntu/jammy.tailscale-keyring.list
+ tee /etc/apt/sources.list.d/tailscale.list
# Tailscale packages for ubuntu jammy
deb [signed-by=/usr/share/keyrings/tailscale-archive-keyring.gpg] https://pkgs.tailscale.com/stable/ubuntu jammy main
+ chmod 0644 /etc/apt/sources.list.d/tailscale.list
+ apt-get update
Hit:1 http://security.ubuntu.com/ubuntu jammy-security InRelease
Hit:2 http://archive.ubuntu.com/ubuntu jammy InRelease
Hit:3 https://developer.download.nvidia.com/compute/cuda/repos/ubuntu2204/x86_64  InRelease
Get:4 https://cli.github.com/packages stable InRelease [3,917 B]
Hit:5 https://cloud.r-project.org/bin/linux/ubuntu jammy-cran40/ InRelease
Hit:6

In [ ]:
from IPython.display import display, HTML

display(HTML("""
<script>
const ctx = new (window.AudioContext || window.webkitAudioContext)();
const merger = ctx.createChannelMerger(2);

const gainL = ctx.createGain();
const gainR = ctx.createGain();
gainL.gain.value = 0.4;
gainR.gain.value = 0.4;

const oscL = ctx.createOscillator();
const oscR = ctx.createOscillator();
oscL.type = 'sine';
oscR.type = 'sine';
oscL.frequency.value = 100;   // oído izquierdo
oscR.frequency.value = 133;   // oído derecho (diferencia = 33 Hz Gamma)

oscL.connect(gainL); gainL.connect(merger, 0, 0);
oscR.connect(gainR); gainR.connect(merger, 0, 1);
merger.connect(ctx.destination);

oscL.start();
oscR.start();
</script>
<small style="color:gray">🎧 Binaural activo — 100 Hz / 133 Hz (Gamma 33 Hz)</small>
"""))
# =========================================================
# COMFYUI + PINGGY TUNNEL
# =========================================================

import subprocess
import threading
import time
import re

# =========================================================
# FUNCIÓN TÚNEL PINGGY
# =========================================================

def run_pinggy():
    print("🌐 Iniciando túnel Pinggy...")

    process = subprocess.Popen(
        [
            "ssh",
            "-p", "443",
            "-o", "StrictHostKeyChecking=no",
            "-R0:localhost:8188",
            "a.pinggy.io"
        ],
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        text=True,
        bufsize=1
    )

    for line in process.stdout:
        print(line.strip())

        match = re.search(r"https://[^\s]+\.a\.free\.pinggy\.link", line)

        if match:
            print("\n" + "=" * 70)
            print("🚀 COMFYUI PUBLIC URL:")
            print(match.group(0))
            print("=" * 70 + "\n")

# =========================================================
# LANZAR TÚNEL
# =========================================================

threading.Thread(
    target=run_pinggy,
    daemon=True
).start()

# =========================================================
# ESPERA
# =========================================================

time.sleep(5)

# =========================================================
# INICIAR COMFYUI
# =========================================================

%cd /content/ComfyUI

!python main.py \
    --listen 0.0.0.0 \
    --port 8188 \
    --dont-print-server

